# MyFirstNEURON — Colab Edition (Prototype 2)

This notebook is a Google-Colab-friendly, Python/Jupyter port of **MyFirstNEURON**, a NEURON demo
by Arthur Houweling and Terry Sejnowski (Salk Institute), based on experiments from
*Electrophysiology of the Neuron* by Huguenard & McCormick. Original files:
https://modeldb.science/3808

**Status:** all 17 presets from the original menus ("Basics" 1-6, "Fast Na,K" 7-10,
"Other" currents 11-15, "Synaptic" 16-17) are implemented as ONE unified dashboard (section 2)
sharing a single cell, a single universal run function, and a single set of plotting/dashboard
helpers (section 1). In the original GUI each menu entry covers one or more textbook experiments,
which is why there are fewer presets than textbook experiment numbers.

No local installation is needed — just run the cells top to bottom in Colab.

# 1. Setup

## 1.1 Installation (run once per Colab session)

Installs the `neuron` Python package, clones the original `.mod` mechanism files from the
[ModelDB GitHub mirror](https://github.com/ModelDBRepository/3808), and compiles them with
`nrnivmodl`.

In [ ]:
%%capture
!pip install neuron

In [ ]:
import os

MOD_SRC_DIR = "mfn_src"

if not os.path.isdir(MOD_SRC_DIR):
    !git clone --depth 1 https://github.com/ModelDBRepository/3808.git {MOD_SRC_DIR}

!cd {MOD_SRC_DIR} && nrnivmodl

In [ ]:
from neuron import n
from neuron import load_mechanisms
import matplotlib.pyplot as plt

load_mechanisms(MOD_SRC_DIR)
n.load_file("stdrun.hoc")

print("NEURON is ready, mechanisms loaded from:", MOD_SRC_DIR)

## 1.2 Build the cell

A single spherical compartment (same geometry as the original demo: total membrane area of
29000 &mu;m&sup2;), with passive leak channels (Na/K/Ca/Cl/Mg) and Hodgkin-Huxley
sodium/potassium channels. All experiments 1-6 reuse this same cell; only the parameter
values differ.

In [ ]:
import math

soma = n.Section(name="soma")
soma.L = 290 / math.pi
soma.diam = 100
soma.nseg = 1

for mech in ("leak", "HH", "iA", "iL", "iT", "iC", "iAHP", "iM", "cadyn"):
    soma.insert(mech)

# calcium-shell decay constants for cadyn, fixed across every experiment (never overridden
# by any e*.par file): a 1 ms relaxation towards 5e-5 mM, pump disabled (kt=0)
soma(0.5).taur_cadyn = 1.0
soma(0.5).cainf_cadyn = 5e-5
soma(0.5).kt_cadyn = 0.0
soma(0.5).kd_cadyn = 5e-5

# celsius is set per-experiment (see _apply_cell_params below); e7.par uses 23.5 instead of 35
print("soma area (um^2):", n.area(0.5, sec=soma))

## 1.3 Common recording

`t_vec`/`v_vec` are shared by every experiment; section 1.4 below adds the current-clamp,
voltage-clamp and synaptic point processes (and their recordings) used by every experiment type.


In [ ]:
t_vec = n.Vector().record(n._ref_t)
v_vec = n.Vector().record(soma(0.5)._ref_v)

## 1.4 Stimulus, voltage-clamp & synaptic objects

Every experiment shares the same handful of point processes: a current-clamp pulse electrode
(`stim`, matching the original hoc's `stim[1]`), a constant "base" current electrode (`stim2`,
matching `stim[0]`), a 3-phase voltage clamp (`vclamp`, matching `stim[2]`), and the four
alpha-synapse point processes. They are all created once, up front, and every run leaves the
unused ones at an inert (zero) default — this is what lets a single, universal run function work
for every experiment (see section 1.5).

In [ ]:
stim = n.IClamp(soma(0.5))       # hoc's stim[1]: current-clamp pulse
i_vec = n.Vector().record(stim._ref_i)

stim2 = n.IClamp(soma(0.5))      # hoc's stim[0]: constant "base" current
i2_vec = n.Vector().record(stim2._ref_i)

vclamp = n.VClamp(soma(0.5))     # hoc's stim[2]: 3-phase voltage clamp
ina_vec = n.Vector().record(soma(0.5)._ref_ina)
ik_vec = n.Vector().record(soma(0.5)._ref_ik)
m_HH_vec = n.Vector().record(soma(0.5)._ref_m_HH)
h_HH_vec = n.Vector().record(soma(0.5)._ref_h_HH)
n_HH_vec = n.Vector().record(soma(0.5)._ref_n_HH)

ampasyn = n.AmpaSynapse(soma(0.5))
nmdasyn = n.NmdaSynapse(soma(0.5))
gabaAsyn = n.GABAaSynapse(soma(0.5))
gabaBsyn = n.GABAbSynapse(soma(0.5))
ampasyn_i_vec = n.Vector().record(ampasyn._ref_i)
nmdasyn_i_vec = n.Vector().record(nmdasyn._ref_i)
gabaAsyn_i_vec = n.Vector().record(gabaAsyn._ref_i)
gabaBsyn_i_vec = n.Vector().record(gabaBsyn._ref_i)

ik_iA_vec = n.Vector().record(soma(0.5)._ref_ik_iA)
ica_iL_vec = n.Vector().record(soma(0.5)._ref_ica_iL)
iCa_iT_vec = n.Vector().record(soma(0.5)._ref_iCa_iT)
ik_iC_vec = n.Vector().record(soma(0.5)._ref_ik_iC)
ik_iAHP_vec = n.Vector().record(soma(0.5)._ref_ik_iAHP)

ik_iM_vec = n.Vector().record(soma(0.5)._ref_ik_iM)
cai_vec = n.Vector().record(soma(0.5)._ref_cai)

## 1.5 Shared simulation parameters

`DEFAULT_PARAMS` holds every value the model needs — cell/ion baseline, current-clamp, voltage-
clamp and synaptic — with inert (zero) defaults for whichever stimuli a given experiment doesn't
use. `_apply_params()` unconditionally applies the *entire* merged parameter dict on every run,
so there is no separate "reset stimuli between experiments" step: switching experiments just
means a different subset of overrides is layered on top of `DEFAULT_PARAMS`.

In [ ]:
# every value the model needs, across every experiment type; inert (zero) defaults for whatever
# stimulus/synapse a given experiment doesn't use
DEFAULT_PARAMS = dict(
    # ionic concentrations & temperature
    celsius=35.0, nai=31.0, nao=145.0, ki=135.0, ko=3.1,
    cli=7.0, clo=120.0, mgo=1.0, cai0_ca_ion=5e-5, cao0_ca_ion=2.0,
    # membrane conductances / permeabilities
    pna_leak=0.0, pk_leak=0.0,
    gnabar_HH=0.069, gkbar_HH=0.0069,
    gkbar_iA=0.0, pcabar_iL=0.0, pcabar_iT=0.0, gkbar_iC=0.0, gkbar_iAHP=0.0, gkbar_iM=0.0,
    # run / timing
    v_init=-65.0, tstop=20.0,
    # current-clamp stimuli (stim = hoc's stim[1], stim2 = hoc's stim[0], the "base" current)
    stim_delay=0.0, stim_dur=0.0, stim_amp=0.0,
    stim2_delay=0.0, stim2_dur=9999.0, stim2_amp=0.0,
    # voltage clamp: 3-phase hold / step / return (hoc's stim[2])
    vc_dur0=0.0, vc_dur1=0.0, vc_dur2=0.0,
    vc_amp0=0.0, vc_amp1=0.0, vc_amp2=0.0,
    # synaptic
    gmax_EPSP=0.0, onset_EPSP=20.0, ampasyn_w=1.0, nmdasyn_w=0.5,
    gmax_IPSP=0.0, onset_IPSP=25.0, gabaAsyn_w=1.0, gabaBsyn_w=0.05,
)


def _apply_params(p):
    """Apply the FULL parameter set to the cell and to every stimulus/synapse object. Every
    run (of every experiment type) calls this with the same {**DEFAULT_PARAMS, **overrides}
    pattern, so unused stimuli always fall back to an inert default instead of needing a
    separate reset step (mirrors reset_soma_pars() in the original hoc, which zeroes
    everything before every experiment)."""
    soma(0.5).pna_leak = p["pna_leak"]
    soma(0.5).pk_leak = p["pk_leak"]
    soma(0.5).gnabar_HH = p["gnabar_HH"]
    soma(0.5).gkbar_HH = p["gkbar_HH"]
    soma(0.5).gkbar_iA = p["gkbar_iA"]
    soma(0.5).pcabar_iL = p["pcabar_iL"]
    soma(0.5).pcabar_iT = p["pcabar_iT"]
    soma(0.5).gkbar_iC = p["gkbar_iC"]
    soma(0.5).gkbar_iAHP = p["gkbar_iAHP"]
    soma(0.5).gkbar_iM = p["gkbar_iM"]
    n.cai0_ca_ion = p["cai0_ca_ion"]
    n.cao0_ca_ion = p["cao0_ca_ion"]
    soma(0.5).nai = p["nai"]
    soma(0.5).nao = p["nao"]
    soma(0.5).ki = p["ki"]
    soma(0.5).ko = p["ko"]
    soma(0.5).cli = p["cli"]
    soma(0.5).clo = p["clo"]
    soma(0.5).mgo = p["mgo"]
    n.celsius = p["celsius"]

    stim.delay, stim.dur, stim.amp = p["stim_delay"], p["stim_dur"], p["stim_amp"]
    stim2.delay, stim2.dur, stim2.amp = p["stim2_delay"], p["stim2_dur"], p["stim2_amp"]

    vclamp.dur[0], vclamp.dur[1], vclamp.dur[2] = p["vc_dur0"], p["vc_dur1"], p["vc_dur2"]
    vclamp.amp[0], vclamp.amp[1], vclamp.amp[2] = p["vc_amp0"], p["vc_amp1"], p["vc_amp2"]

    ampasyn.gmaxEPSP = nmdasyn.gmaxEPSP = p["gmax_EPSP"]
    ampasyn.onset = nmdasyn.onset = p["onset_EPSP"]
    ampasyn.w = p["ampasyn_w"]
    nmdasyn.w = p["nmdasyn_w"]
    gabaAsyn.gmaxIPSP = gabaBsyn.gmaxIPSP = p["gmax_IPSP"]
    gabaAsyn.onset = gabaBsyn.onset = p["onset_IPSP"]
    gabaAsyn.w = p["gabaAsyn_w"]
    gabaBsyn.w = p["gabaBsyn_w"]

## 1.6 Panel groups & plotting core

`PANEL_GROUPS` is a code-level configuration: each entry is one subplot, and its `signals` list
controls which traces are overlaid on that subplot (edit this dict to show other traces — e.g.
add another vector to `"K_currents"` to compare it against iA/iC/iAHP/iM). `run_once()` is the
single, universal run function used by *every* experiment. `run_series()` reruns the model while
sweeping `vc_amp1`, overlaying every step's traces on the same axes — a generalization of the
original hoc's `series_voltagestep()` "Run Series" button.


In [ ]:
PANEL_GROUPS = {
    "v": dict(label="membrane potential", ylabel="v (mV)", y_min=-100.0, y_max=50.0,
              signals=[("v", lambda: v_vec)]),
    "i_inj": dict(label="injected current", ylabel="i (nA)", y_min=-2.0, y_max=2.0,
                  signals=[("stim", lambda: i_vec), ("stim2", lambda: i2_vec)]),
    "im": dict(label="ina + ik (vclamp)", ylabel="i (mA/cm2)", y_min=-1.0, y_max=1.0,
               signals=[("ina+ik", lambda: ina_vec.c().add(ik_vec))]),
    "gating": dict(label="Na/K gating variables (exp. 10)", ylabel="", y_min=0.0, y_max=1.0,
                    signals=[("m", lambda: m_HH_vec), ("h", lambda: h_HH_vec), ("n", lambda: n_HH_vec)]),
    "HH_conductances": dict(label="gNa*m^3*h & gK*n^4 (exp. 10)", ylabel="g (S/cm2)", y_min=0.0, y_max=0.006,
                             signals=[("gNa", lambda: m_HH_vec.c().pow(3).mul(h_HH_vec).mul(soma(0.5).gnabar_HH)),
                                      ("gK", lambda: n_HH_vec.c().pow(4).mul(soma(0.5).gkbar_HH))]),
    "K_currents": dict(label="K currents (iA/iC/iAHP/iM)", ylabel="i (mA/cm2)", y_min=-0.02, y_max=0.02,
                        signals=[("iA", lambda: ik_iA_vec), ("iC", lambda: ik_iC_vec),
                                 ("iAHP", lambda: ik_iAHP_vec), ("iM", lambda: ik_iM_vec)]),
    "Ca_currents": dict(label="Ca currents (iL/iT)", ylabel="i (mA/cm2)", y_min=-0.05, y_max=0.01,
                          signals=[("iL", lambda: ica_iL_vec), ("iT", lambda: iCa_iT_vec)]),
    "cai": dict(label="[Ca2+]i", ylabel="cai (mM)", y_min=0.0, y_max=0.02,
                signals=[("cai", lambda: cai_vec)]),
    "isyn": dict(label="synaptic currents", ylabel="i (nA)", y_min=-0.5, y_max=0.5,
                 signals=[("ampa", lambda: ampasyn_i_vec), ("nmda", lambda: nmdasyn_i_vec),
                          ("gabaA", lambda: gabaAsyn_i_vec), ("gabaB", lambda: gabaBsyn_i_vec)]),
}


def _signal_values(signal):
    # every signal entry is a zero-arg callable (NEURON Vector objects are themselves
    # callable via hoc's x(i) indexing convention, so callable() can't distinguish them)
    return list(signal())


def _capture_run(keys):
    """Snapshot t_vec + every signal of the given panel groups right after a run() call — the
    underlying NEURON Vectors get overwritten in place by the next run(), so a copy of the
    data must be taken before then (needed for run_series() to overlay multiple runs)."""
    t_list = list(t_vec)
    return {key: (t_list, [(label, _signal_values(sig)) for label, sig in PANEL_GROUPS[key]["signals"]])
            for key in keys}


def _render_groups(keys, y_ranges, runs):
    """runs: list of (run_label, snapshot) pairs to overlay on the same axes. A single run
    passes one entry with run_label=None; run_series() passes one entry per swept value."""
    if not keys:
        print("No panels selected.")
        return

    fig, axes = plt.subplots(len(keys), 1, figsize=(8, 2.6 * len(keys)), sharex=True)
    axes = [axes] if len(keys) == 1 else list(axes)
    for ax, key in zip(axes, keys):
        spec = PANEL_GROUPS[key]
        show_legend = len(spec["signals"]) > 1 or any(run_label is not None for run_label, _ in runs)
        for run_label, snapshot in runs:
            t_list, signals = snapshot[key]
            for sig_label, y_list in signals:
                if run_label is None:
                    label = sig_label
                elif len(signals) > 1:
                    label = f"{sig_label} ({run_label})"
                else:
                    label = run_label
                ax.plot(t_list, y_list, label=label)
        ax.set_ylabel(spec["ylabel"])
        y_min, y_max = y_ranges.get(key, (spec.get("y_min"), spec.get("y_max")))
        if y_min is not None and y_max is not None:
            ax.set_ylim(y_min, y_max)
        if show_legend:
            ax.legend(fontsize=8, loc="upper right")
    axes[-1].set_xlabel("time (ms)")
    plt.tight_layout()
    plt.show()


def run_once(keys, y_ranges, overrides):
    """THE universal run function, used by every experiment (current-clamp, voltage-clamp,
    other-currents, synaptic alike)."""
    p = {**DEFAULT_PARAMS, **overrides}
    _apply_params(p)
    n.v_init = p["v_init"]
    n.tstop = p["tstop"]
    n.run()
    _render_groups(keys, y_ranges, [(None, _capture_run(keys))])


def run_series(keys, y_ranges, overrides, start, stop, steps):
    """Sweeps vc_amp1 from start to stop over `steps` runs, overlaying every run's traces on
    the same axes with a legend — generalizes the original hoc's series_voltagestep()."""
    steps = max(int(steps), 1)
    runs = []
    for i in range(steps):
        value = start if steps == 1 else start + (stop - start) * i / (steps - 1)
        p = {**DEFAULT_PARAMS, **overrides, "vc_amp1": value}
        _apply_params(p)
        n.v_init = p["v_init"]
        n.tstop = p["tstop"]
        n.run()
        runs.append((f"amp1={value:g}mV", _capture_run(keys)))
    _render_groups(keys, y_ranges, runs)


## 1.7 Shared dashboard helpers

Every parameter is a plain numeric text box (`FloatText`, no slider) grouped into collapsible
`Accordion` sections (ionic concentrations, membrane conductances, run/timing, current-clamp
stimuli, voltage clamp, synaptic). `_build_panel_checks()` builds the panel-visibility checkboxes
plus a persistent y-min/y-max text box pair per panel, so repeated runs stay visually comparable.


In [ ]:
from ipywidgets import (FloatText, Checkbox, Button, Dropdown, Accordion, BoundedIntText,
                         HBox, VBox, Output, Layout, Label)
from IPython.display import display

GROUP_LABELS = {
    "ions": "Ionic concentrations & temperature",
    "conductances": "Membrane conductances",
    "run": "Run / timing",
    "stim": "Current-clamp stimuli",
    "vclamp": "Voltage clamp",
    "synaptic": "Synaptic",
}


def _build_param_rows(param_ui, get_default):
    """get_default(name) -> current default value for that parameter (changes with the
    selected experiment preset)."""
    fields, changed_flags, rows_by_group = {}, {}, {}

    def _make_change_handler(name):
        def _on_change(change):
            changed_flags[name].value = (change["new"] != get_default(name))
        return _on_change

    def _make_reset_handler(name):
        def _on_click(_btn):
            fields[name].value = get_default(name)
        return _on_click

    for name, spec in param_ui.items():
        field = FloatText(value=get_default(name), step=spec.get("step", 1), description=spec["description"],
                           layout=Layout(width="240px"), style={"description_width": "120px"})
        unit_label = Label(value=spec.get("unit", ""), layout=Layout(width="45px"))
        changed = Checkbox(value=False, description="changed", disabled=True, indent=False,
                            layout=Layout(width="85px"))
        reset_btn = Button(description="reset", layout=Layout(width="55px"))

        fields[name] = field
        changed_flags[name] = changed
        field.observe(_make_change_handler(name), names="value")
        reset_btn.on_click(_make_reset_handler(name))

        rows_by_group.setdefault(spec.get("group", "other"), []).append(
            HBox([field, unit_label, changed, reset_btn]))

    return fields, changed_flags, rows_by_group


def _build_param_accordion(param_ui, get_default):
    fields, changed_flags, rows_by_group = _build_param_rows(param_ui, get_default)
    group_keys = list(dict.fromkeys(spec.get("group", "other") for spec in param_ui.values()))
    accordion = Accordion(children=[VBox(rows_by_group[g]) for g in group_keys])
    for i, g in enumerate(group_keys):
        accordion.set_title(i, GROUP_LABELS.get(g, g))
    return fields, changed_flags, accordion


def _build_panel_checks(panel_groups):
    checks, y_fields, rows = {}, {}, []
    for key, spec in panel_groups.items():
        cb = Checkbox(value=True, description=spec["label"], indent=False, layout=Layout(width="230px"))
        y_min = FloatText(value=spec.get("y_min"), description="y-min", layout=Layout(width="130px"),
                           style={"description_width": "40px"})
        y_max = FloatText(value=spec.get("y_max"), description="y-max", layout=Layout(width="130px"),
                           style={"description_width": "40px"})
        checks[key] = cb
        y_fields[key] = (y_min, y_max)
        rows.append(HBox([cb, y_min, y_max]))
    return checks, y_fields, VBox([Label("Panels (check to show; set the y-axis range next to each):")] + rows)


def _selected_panels(panel_checks):
    return [key for key, cb in panel_checks.items() if cb.value]


def _y_ranges(y_fields, keys):
    return {key: (y_fields[key][0].value, y_fields[key][1].value) for key in keys}


def _field_values(fields):
    return {name: field.value for name, field in fields.items()}


# 2. Experiments

One dashboard covers all 17 presets (spanning textbook experiments 1-17). Pick a preset from the
dropdown to load its defaults, tweak any parameter (grouped into collapsible sections below), and
press **Run**. Presets 4, 7, 12 and 15 use the voltage clamp; for those, **Run series** sweeps
`vc_amp1` (the voltage-clamp step potential) from *start* to *stop* and overlays every step on the
same axes — the notebook equivalent of the original hoc's "Run Series" button.


In [ ]:
# each entry overrides only the DEFAULT_PARAMS keys that differ from baseline; original .par
# file used in a comment for traceability
EXPERIMENTS = {
    "1: Resting potential (exp. 1, 2)": dict(  # e1.par
        pna_leak=2.07e-07, pk_leak=3.45e-06,
        v_init=-65.0, tstop=20.0,
    ),
    "2: Membrane properties (exp. 3, 4)": dict(  # e3.par
        gnabar_HH=0.069, gkbar_HH=0.0069, pna_leak=2.07e-07, pk_leak=3.45e-06,
        v_init=-65.0, tstop=80.0, stim_delay=10.0, stim_dur=50.0, stim_amp=1.5,
    ),
    "3: Impulse generation (exp. 5, 6)": dict(  # e5.par
        gnabar_HH=0.069, gkbar_HH=0.0069, pna_leak=2.07e-07, pk_leak=3.45e-06,
        v_init=-65.0, tstop=80.0, stim_delay=10.0, stim_dur=50.0, stim_amp=2.0,
    ),
    "4: Voltage clamp (exp. 7, 8, 9)": dict(  # e7.par
        celsius=23.5, gnabar_HH=0.0345, gkbar_HH=0.0069, nai=30.0,
        v_init=-65.0, tstop=20.0,
        vc_dur0=10.0, vc_dur1=10.0, vc_amp0=-100.0, vc_amp1=0.0, vc_amp2=-100.0,
    ),
    "5: Na+/K+ gating variables (exp. 10)": dict(  # e10.par
        gnabar_HH=0.069, gkbar_HH=0.0069, pna_leak=4.14e-07, pk_leak=3.45e-06,
        nai=31.0, nao=145.0, ki=135.0, ko=3.1,
        v_init=-50.0, tstop=15.0,
    ),
    "6: iA - action potential (exp. 11)": dict(  # e11a.par
        pna_leak=6.9e-08, pk_leak=4.14e-07, gnabar_HH=0.0517, gkbar_HH=0.0069, gkbar_iA=0.00345,
        v_init=-65.0, tstop=60.0,
    ),
    "7: iA - voltage clamp (exp. 11)": dict(  # e11b.par
        gkbar_iA=0.00345, nai=30.0, celsius=23.5,
        v_init=-100.0, tstop=100.0,
        vc_dur0=10.0, vc_dur1=100.0, vc_amp0=-100.0,
    ),
    "8: iL & iC (exp. 12)": dict(  # e12.par
        gnabar_HH=0.0517, gkbar_HH=0.00345, gkbar_iC=0.00345, pcabar_iL=0.000276,
        pna_leak=4.31e-08, pk_leak=4.14e-07,
        v_init=-55.0, tstop=30.0, stim_delay=3.0, stim_dur=3.0, stim_amp=1.0,
    ),
    "9: iAHP (exp. 13)": dict(  # e13.par
        pna_leak=2e-08, pk_leak=4.14e-07, pcabar_iL=0.000276, gnabar_HH=0.0517, gkbar_HH=0.0069,
        gkbar_iAHP=0.000207, gkbar_iC=0.00345,
        v_init=-70.0, tstop=600.0, stim_delay=50.0, stim_dur=300.0, stim_amp=0.6,
    ),
    "10: iT (exp. 14)": dict(  # e14.par
        pna_leak=2.7414e-08, pk_leak=5.069e-07, pcabar_iL=0.00027586, pcabar_iT=0.00010345,
        gkbar_iC=0.0068966, gkbar_iA=0.0034483, gnabar_HH=0.051724, gkbar_HH=0.0068966,
        v_init=-85.0, tstop=300.0,
        stim2_amp=-0.27,
        stim_delay=25.0, stim_dur=200.0, stim_amp=0.12,
    ),
    "11: iM - current clamp (exp. 15)": dict(  # e15a.par
        pna_leak=1.7241e-08, pk_leak=4.1379e-07, gkbar_iM=0.00031035, gnabar_HH=0.051724, gkbar_HH=0.0068966,
        v_init=-75.0, tstop=300.0, stim_delay=50.0, stim_dur=150.0, stim_amp=0.7,
    ),
    "12: iM - voltage clamp (exp. 15)": dict(  # e15b.par
        pna_leak=3.4483e-08, pk_leak=3.4483e-07, gkbar_iM=0.00086207, nai=30.0,
        v_init=-70.0, tstop=400.0,
        vc_dur0=50.0, vc_dur1=250.0, vc_dur2=100.0, vc_amp0=-70.0, vc_amp1=-30.0, vc_amp2=-70.0,
    ),
    "13: e.p.s.p. (exp. 16)": dict(  # e16a.par
        pna_leak=3.4138e-08, pk_leak=3.4483e-07,
        gmax_EPSP=0.02, onset_EPSP=20.0, ampasyn_w=1.0, nmdasyn_w=0.5,
        gmax_IPSP=0.1, onset_IPSP=25.0, gabaAsyn_w=1.0, gabaBsyn_w=0.05,
        mgo=1.2, v_init=-55.0, tstop=500.0,
    ),
    "14: nmda current (exp. 16)": dict(  # e16b.par
        pna_leak=4.069e-08, pk_leak=4.1379e-07,
        gmax_EPSP=0.02, onset_EPSP=10.0, ampasyn_w=0.0, nmdasyn_w=1.0,
        gmax_IPSP=0.0, onset_IPSP=25.0, gabaAsyn_w=1.0, gabaBsyn_w=0.05,
        mgo=1.2, v_init=-55.0, tstop=120.0,
    ),
    "15: nmda voltage clamp (exp. 16)": dict(  # e16c.par
        nai=30.0, cli=8.0, clo=140.0,
        vc_dur0=2.0, vc_dur1=28.0, vc_dur2=0.0, vc_amp0=-100.0, vc_amp1=-20.0, vc_amp2=0.0,
        gmax_EPSP=0.1, onset_EPSP=1.0, ampasyn_w=0.0, nmdasyn_w=1.0,
        gmax_IPSP=0.0, onset_IPSP=25.0, gabaAsyn_w=1.0, gabaBsyn_w=0.05,
        v_init=-55.0, tstop=30.0,
    ),
    "16: i.p.s.p. (exp. 17)": dict(  # e17a.par
        pna_leak=3.3793e-08, pk_leak=3.4483e-07,
        gmax_EPSP=0.0, onset_EPSP=20.0, ampasyn_w=1.0, nmdasyn_w=0.5,
        gmax_IPSP=0.1, onset_IPSP=25.0, gabaAsyn_w=1.0, gabaBsyn_w=0.05,
        mgo=1.2, v_init=-55.0, tstop=500.0,
    ),
    "17: i.p.s.p. + e.p.s.p. (exp. 17)": dict(  # e17b.par
        pna_leak=6.0345e-08, pk_leak=5.5172e-07, gnabar_HH=0.051724, gkbar_HH=0.0068966, gkbar_iA=0.0034483,
        stim2_amp=-0.69,
        gmax_EPSP=0.15, onset_EPSP=20.0, ampasyn_w=1.0, nmdasyn_w=1.0,
        gmax_IPSP=0.0, onset_IPSP=21.0, gabaAsyn_w=1.0, gabaBsyn_w=0.0,
        v_init=-85.0, tstop=100.0,
    ),
}

PARAM_UI = {
    # ---- ionic concentrations & temperature ----
    "celsius": dict(description="celsius", unit="degC", step=0.5, group="ions"),
    "nai": dict(description="nai", unit="mM", step=1, group="ions"),
    "nao": dict(description="nao", unit="mM", step=1, group="ions"),
    "ki": dict(description="ki", unit="mM", step=1, group="ions"),
    "ko": dict(description="ko", unit="mM", step=0.1, group="ions"),
    "cli": dict(description="cli", unit="mM", step=0.5, group="ions"),
    "clo": dict(description="clo", unit="mM", step=1, group="ions"),
    "mgo": dict(description="mgo", unit="mM", step=0.1, group="ions"),
    "cai0_ca_ion": dict(description="cai0", unit="mM", step=1e-5, group="ions"),
    "cao0_ca_ion": dict(description="cao0", unit="mM", step=0.1, group="ions"),
    # ---- membrane conductances / permeabilities ----
    "pna_leak": dict(description="pNa (leak)", unit="cm/s", step=1e-8, group="conductances"),
    "pk_leak": dict(description="pK (leak)", unit="cm/s", step=1e-7, group="conductances"),
    "gnabar_HH": dict(description="gNa (HH)", unit="S/cm2", step=0.001, group="conductances"),
    "gkbar_HH": dict(description="gK (HH)", unit="S/cm2", step=0.0005, group="conductances"),
    "gkbar_iA": dict(description="gK (iA)", unit="S/cm2", step=0.0005, group="conductances"),
    "pcabar_iL": dict(description="pCa (iL)", unit="cm/s", step=1e-5, group="conductances"),
    "pcabar_iT": dict(description="pCa (iT)", unit="cm/s", step=1e-5, group="conductances"),
    "gkbar_iC": dict(description="gK (iC)", unit="S/cm2", step=0.0005, group="conductances"),
    "gkbar_iAHP": dict(description="gK (iAHP)", unit="S/cm2", step=1e-5, group="conductances"),
    "gkbar_iM": dict(description="gK (iM)", unit="S/cm2", step=1e-5, group="conductances"),
    # ---- run / timing ----
    "v_init": dict(description="v_init", unit="mV", step=1, group="run"),
    "tstop": dict(description="tstop", unit="ms", step=5, group="run"),
    # ---- current-clamp stimuli ----
    "stim_delay": dict(description="stim delay", unit="ms", step=1, group="stim"),
    "stim_dur": dict(description="stim dur", unit="ms", step=1, group="stim"),
    "stim_amp": dict(description="stim amp", unit="nA", step=0.01, group="stim"),
    "stim2_delay": dict(description="stim2 delay", unit="ms", step=1, group="stim"),
    "stim2_dur": dict(description="stim2 dur", unit="ms", step=1, group="stim"),
    "stim2_amp": dict(description="stim2 amp", unit="nA", step=0.01, group="stim"),
    # ---- voltage clamp ----
    "vc_dur0": dict(description="dur0 (hold)", unit="ms", step=1, group="vclamp"),
    "vc_dur1": dict(description="dur1 (step)", unit="ms", step=1, group="vclamp"),
    "vc_dur2": dict(description="dur2 (return)", unit="ms", step=1, group="vclamp"),
    "vc_amp0": dict(description="amp0 (hold)", unit="mV", step=1, group="vclamp"),
    "vc_amp1": dict(description="amp1 (step)", unit="mV", step=1, group="vclamp"),
    "vc_amp2": dict(description="amp2 (return)", unit="mV", step=1, group="vclamp"),
    # ---- synaptic ----
    "gmax_EPSP": dict(description="gmax EPSP", unit="nS", step=0.005, group="synaptic"),
    "onset_EPSP": dict(description="onset EPSP", unit="ms", step=1, group="synaptic"),
    "ampasyn_w": dict(description="ampasyn.w", unit="", step=0.05, group="synaptic"),
    "nmdasyn_w": dict(description="nmdasyn.w", unit="", step=0.05, group="synaptic"),
    "gmax_IPSP": dict(description="gmax IPSP", unit="nS", step=0.005, group="synaptic"),
    "onset_IPSP": dict(description="onset IPSP", unit="ms", step=1, group="synaptic"),
    "gabaAsyn_w": dict(description="gabaAsyn.w", unit="", step=0.05, group="synaptic"),
    "gabaBsyn_w": dict(description="gabaBsyn.w", unit="", step=0.05, group="synaptic"),
}


In [ ]:
def build_experiment_dashboard():
    current_defaults = {**DEFAULT_PARAMS, **EXPERIMENTS[next(iter(EXPERIMENTS))]}

    exp_selector = Dropdown(options=list(EXPERIMENTS.keys()), description="Experiment:",
                             style={"description_width": "initial"}, layout=Layout(width="380px"))

    panel_checks, y_fields, panels_box = _build_panel_checks(PANEL_GROUPS)
    fields, changed_flags, accordion = _build_param_accordion(PARAM_UI, lambda name: current_defaults[name])

    run_button = Button(description="Run", button_style="success", icon="play")
    reset_all_button = Button(description="Reset all", icon="undo")
    output = Output()

    series_start = FloatText(value=-100.0, description="start", layout=Layout(width="150px"),
                              style={"description_width": "45px"})
    series_stop = FloatText(value=50.0, description="stop", layout=Layout(width="150px"),
                             style={"description_width": "45px"})
    series_steps = BoundedIntText(value=6, min=2, max=30, description="steps", layout=Layout(width="150px"),
                                   style={"description_width": "45px"})
    series_button = Button(description="Run series (vc_amp1)", icon="bars")
    series_row = HBox([series_start, series_stop, series_steps, series_button])

    def _on_run(_btn):
        with output:
            output.clear_output(wait=True)
            keys = _selected_panels(panel_checks)
            y_ranges = _y_ranges(y_fields, keys)
            run_once(keys, y_ranges, _field_values(fields))

    def _on_run_series(_btn):
        with output:
            output.clear_output(wait=True)
            keys = _selected_panels(panel_checks)
            y_ranges = _y_ranges(y_fields, keys)
            run_series(keys, y_ranges, _field_values(fields), series_start.value, series_stop.value, series_steps.value)

    def _on_reset_all(_btn):
        for name, field in fields.items():
            field.value = current_defaults[name]

    def _on_experiment_change(change):
        nonlocal current_defaults
        current_defaults = {**DEFAULT_PARAMS, **EXPERIMENTS[change["new"]]}
        _on_reset_all(None)
        _on_run(None)

    run_button.on_click(_on_run)
    series_button.on_click(_on_run_series)
    reset_all_button.on_click(_on_reset_all)
    exp_selector.observe(_on_experiment_change, names="value")

    dashboard = VBox([
        exp_selector,
        HBox([
            VBox([panels_box, series_row, output], layout=Layout(width="640px")),
            VBox([HBox([run_button, reset_all_button]), accordion], layout=Layout(width="420px")),
        ]),
    ])
    _on_run(None)
    return dashboard


## Usage

1. Pick an experiment from the dropdown — its parameter values load into the text boxes below
   and it runs automatically.
2. Edit any text box (grouped by category — expand a section to see its parameters) and press
   **Run**; the "changed" checkbox next to a field flags it as different from the preset, and
   **reset** restores just that one value. **Reset all** restores every field.
3. Check/uncheck panels to show/hide them, and edit the y-min/y-max boxes next to each panel to
   keep the y-axis fixed across runs (handy for comparing traces after changing a parameter).
4. For voltage-clamp presets (4, 7, 12, 15), use **Run series** to sweep `vc_amp1` from *start*
   to *stop* over *steps* runs, overlaying every step's traces on the same axes.


In [ ]:
display(build_experiment_dashboard())
